In [14]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Phase 3 - Threshold Optimization")
print("="*80)
print("\nLibraries imported successfully")


Phase 3 - Threshold Optimization

Libraries imported successfully


In [15]:
# Cell 2: Directory Configuration

BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')

# Input paths
PREDICTIONS_DIR = BASE_DIR / 'data/processed/Phase 3 - Model Training'
VALIDATION_SUSPICIOUS_DIR = BASE_DIR / 'data/validation/raw'

# Output paths
OUTPUT_DIR = BASE_DIR / 'data/processed/Phase 3 - Model Training'

print("Directories configured")
print(f"Predictions: {PREDICTIONS_DIR}")
print(f"Ground truth: {VALIDATION_SUSPICIOUS_DIR}")
print(f"Output: {OUTPUT_DIR}")


Directories configured
Predictions: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training
Ground truth: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/raw
Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training


In [16]:
# Cell 3: Load Ground Truth from Suspicious CSVs

print("\n" + "="*80)
print("LOADING GROUND TRUTH")
print("="*80)

# Ground truth: Which files are actually timestomped
# Source: Suspicious CSVs from validation datasets

validation_datasets = {
    'LoneWolf': 'LoneWolf',
    '02-APT19': '02-APT19',
    '09-APT40': '09-APT40',
    '12-Kimsuky': '12-Kimsuky',
    '13-Winnti731': '13-Winnti731'
}

# Path to suspicious CSVs
SUSPICIOUS_DIR = BASE_DIR / 'data/validation/suspicious'

ground_truth_files = {}

for dataset_key in validation_datasets.keys():
    print(f"\n{dataset_key}:")
    
    # Load Suspicious CSV
    suspicious_path = SUSPICIOUS_DIR / f'{dataset_key}-Suspicious.csv'
    
    if suspicious_path.exists():
        suspicious_df = pd.read_csv(suspicious_path, encoding='utf-8-sig')
        print(f"  Loaded: {len(suspicious_df)} suspicious events")
        
        # Extract filenames from detail field
        # Detail contains filename in format: "timestamp of filename"
        filenames = set()
        for detail in suspicious_df['detail'].values:
            # Extract filename between "of " and next quote
            if 'of ' in str(detail):
                parts = str(detail).split('of ')
                if len(parts) > 1:
                    # Get text after "of " and before any quote or space
                    fname_part = parts[1].split('"')[0].strip()
                    # Clean up the filename
                    fname = fname_part.replace('"', '').strip()
                    if fname and not fname.startswith('['):
                        filenames.add(fname)
        
        ground_truth_files[dataset_key] = filenames
        
        print(f"  Timestomped files: {len(filenames)}")
        print(f"  Files: {', '.join(sorted(filenames))}")
    else:
        print(f"  WARNING: Suspicious CSV not found at {suspicious_path}")
        ground_truth_files[dataset_key] = set()

total_gt_files = sum(len(files) for files in ground_truth_files.values())
print(f"\n{'='*80}")
print(f"Total ground truth timestomped files: {total_gt_files}")
print(f"{'='*80}")



LOADING GROUND TRUTH

LoneWolf:
  Loaded: 15 suspicious events
  Timestomped files: 13
  Files: AIRPORT INFORMATION.docx, BladeofGrass.jpg, CubaDearmed.jpg, DarkWolf.png, DeathToll.jpg, DemLogic.jpg, HoldMyTidePod.jpg, Huckleberry.png, MyTiredHead.jpg, Planning.docx, RedGuns.jpg, Sheep.jpg, Typical Data Wiping : Deletion after Renaming and Timestamp Manipulation

02-APT19:
  Loaded: 3 suspicious events
  Timestomped files: 1
  Files: 2136906.tmp

09-APT40:
  Loaded: 3 suspicious events
  Timestomped files: 1
  Files: 618031_tep.dll

12-Kimsuky:
  Loaded: 6 suspicious events
  Timestomped files: 3
  Files: boof.dll, boof.exe, boof.sys

13-Winnti731:
  Loaded: 2 suspicious events
  Timestomped files: 1
  Files: _418390281__.tmp

Total ground truth timestomped files: 19


In [17]:
# Cell 4: Load Validation Predictions

print("\n" + "="*80)
print("LOADING VALIDATION PREDICTIONS")
print("="*80)

validation_predictions = {}

for dataset_key in validation_datasets.keys():
    print(f"\n{dataset_key}:")
    
    # Load predictions from Phase 3
    pred_path = PREDICTIONS_DIR / f'{dataset_key}_predictions.csv'
    
    if pred_path.exists():
        pred_df = pd.read_csv(pred_path, low_memory=False)
        validation_predictions[dataset_key] = pred_df
        
        print(f"  Loaded: {len(pred_df):,} events")
        print(f"  Unique files: {pred_df['filename'].nunique()}")
        print(f"  Mean probability: {pred_df['prediction_probability'].mean():.4f}")
        print(f"  Max probability: {pred_df['prediction_probability'].max():.4f}")
    else:
        print(f"  ERROR: Predictions not found at {pred_path}")

print(f"\nPredictions loaded for {len(validation_predictions)} datasets")



LOADING VALIDATION PREDICTIONS

LoneWolf:
  Loaded: 34,300 events
  Unique files: 7428
  Mean probability: 0.0002
  Max probability: 0.1458

02-APT19:
  Loaded: 23,803 events
  Unique files: 4481
  Mean probability: 0.0005
  Max probability: 0.3575

09-APT40:
  Loaded: 23,519 events
  Unique files: 4497
  Mean probability: 0.0003
  Max probability: 0.2118

12-Kimsuky:
  Loaded: 17,521 events
  Unique files: 3495
  Mean probability: 0.0010
  Max probability: 0.9998

13-Winnti731:
  Loaded: 14,182 events
  Unique files: 2894
  Mean probability: 0.0004
  Max probability: 0.6729

Predictions loaded for 5 datasets


In [18]:
# Cell 5: File-Level Detection Function (FIXED)

def calculate_file_level_detection(predictions_df, ground_truth_filenames, threshold):
    """
    Calculate file-level detection metrics.
    
    A file is considered "detected" if ANY of its events have probability >= threshold.
    
    Args:
        predictions_df: DataFrame with predictions and probabilities
        ground_truth_filenames: Set of filenames that are actually timestomped
        threshold: Classification threshold
    
    Returns:
        detected_files: Set of filenames that were flagged
        true_positives: Set of correctly detected timestomped files
        false_positives: Set of incorrectly flagged benign files
        false_negatives: Set of missed timestomped files
    """
    
    # Apply threshold to get binary predictions
    predictions_df = predictions_df.copy()
    predictions_df['predicted_suspicious'] = (
        predictions_df['prediction_probability'] >= threshold
    )
    
    # Get files that have at least one event predicted as suspicious
    flagged_events = predictions_df[predictions_df['predicted_suspicious']]
    detected_files = set(flagged_events['filename'].dropna().unique())
    
    # Calculate true positives, false positives, false negatives
    true_positives = detected_files & ground_truth_filenames
    false_positives = detected_files - ground_truth_filenames
    false_negatives = ground_truth_filenames - detected_files
    
    return detected_files, true_positives, false_positives, false_negatives

print("File-level detection function defined (FIXED)")


File-level detection function defined (FIXED)


In [19]:
# Cell 6: Test Multiple Thresholds

print("\n" + "="*80)
print("THRESHOLD OPTIMIZATION")
print("="*80)

# Test thresholds from 0.05 to 0.50
thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

results = []
detailed_results = []

for threshold in thresholds:
    print(f"\n{'='*80}")
    print(f"THRESHOLD: {threshold:.2f}")
    print(f"{'='*80}")
    
    total_detected = 0
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    total_ground_truth = 0
    
    for dataset_key in validation_datasets.keys():
        if dataset_key not in validation_predictions:
            continue
        
        pred_df = validation_predictions[dataset_key].copy()
        gt_files = ground_truth_files[dataset_key]
        
        detected, tp, fp, fn = calculate_file_level_detection(
            pred_df, gt_files, threshold
        )
        
        total_detected += len(detected)
        total_true_positives += len(tp)
        total_false_positives += len(fp)
        total_false_negatives += len(fn)
        total_ground_truth += len(gt_files)
        
        print(f"\n{dataset_key}:")
        print(f"  Ground truth files: {len(gt_files)}")
        print(f"  Detected files: {len(detected)}")
        print(f"  True positives: {len(tp)}")
        print(f"  False positives: {len(fp)}")
        print(f"  False negatives: {len(fn)}")
        
        if len(tp) > 0:
            print(f"  Correctly detected: {', '.join(sorted(tp))}")
        if len(fn) > 0:
            print(f"  MISSED: {', '.join(sorted(fn))}")
        if len(fp) > 0 and len(fp) <= 5:
            print(f"  False alarms: {', '.join(sorted(fp))}")
        
        detailed_results.append({
            'threshold': threshold,
            'dataset': dataset_key,
            'ground_truth': len(gt_files),
            'detected': len(detected),
            'true_positives': len(tp),
            'false_positives': len(fp),
            'false_negatives': len(fn)
        })
    
    # Calculate overall metrics
    recall = total_true_positives / total_ground_truth if total_ground_truth > 0 else 0
    precision = total_true_positives / total_detected if total_detected > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n{'-'*80}")
    print(f"OVERALL RESULTS AT THRESHOLD {threshold:.2f}")
    print(f"{'-'*80}")
    print(f"Total ground truth files: {total_ground_truth}")
    print(f"Total files detected: {total_detected}")
    print(f"True positives: {total_true_positives}")
    print(f"False positives: {total_false_positives}")
    print(f"False negatives: {total_false_negatives}")
    print(f"Recall: {recall:.3f} ({recall*100:.1f}%)")
    print(f"Precision: {precision:.3f} ({precision*100:.1f}%)")
    print(f"F1-Score: {f1:.3f}")
    
    results.append({
        'threshold': threshold,
        'ground_truth': total_ground_truth,
        'detected': total_detected,
        'true_positives': total_true_positives,
        'false_positives': total_false_positives,
        'false_negatives': total_false_negatives,
        'recall': recall,
        'precision': precision,
        'f1_score': f1
    })

print("\n" + "="*80)
print("THRESHOLD OPTIMIZATION COMPLETE")
print("="*80")


SyntaxError: unterminated string literal (detected at line 96) (1461185328.py, line 96)

In [ ]:
# Cell 7: Results Summary

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

results_df = pd.DataFrame(results)

print("\nPerformance Across Thresholds:")
print(results_df.to_string(index=False))

# Save results
threshold_results_path = OUTPUT_DIR / 'threshold_optimization_results.csv'
results_df.to_csv(threshold_results_path, index=False)

detailed_results_path = OUTPUT_DIR / 'threshold_optimization_detailed.csv'
pd.DataFrame(detailed_results).to_csv(detailed_results_path, index=False)

print(f"\nResults saved:")
print(f"  {threshold_results_path}")
print(f"  {detailed_results_path}")

# Recommendations
print("\n" + "="*80)
print("RECOMMENDATIONS")
print("="*80)

# Best recall
best_recall_idx = results_df['recall'].idxmax()
best_recall_row = results_df.loc[best_recall_idx]

print(f"\nMaximum Recall:")
print(f"  Threshold: {best_recall_row['threshold']:.2f}")
print(f"  Recall: {best_recall_row['recall']:.3f} ({best_recall_row['recall']*100:.1f}%)")
print(f"  Precision: {best_recall_row['precision']:.3f} ({best_recall_row['precision']*100:.1f}%)")
print(f"  F1-Score: {best_recall_row['f1_score']:.3f}")
print(f"  Files detected: {best_recall_row['true_positives']}/{best_recall_row['ground_truth']}")

# Best F1
best_f1_idx = results_df['f1_score'].idxmax()
best_f1_row = results_df.loc[best_f1_idx]

print(f"\nBest F1-Score:")
print(f"  Threshold: {best_f1_row['threshold']:.2f}")
print(f"  Recall: {best_f1_row['recall']:.3f} ({best_f1_row['recall']*100:.1f}%)")
print(f"  Precision: {best_f1_row['precision']:.3f} ({best_f1_row['precision']*100:.1f}%)")
print(f"  F1-Score: {best_f1_row['f1_score']:.3f}")
print(f"  Files detected: {best_f1_row['true_positives']}/{best_f1_row['ground_truth']}")



RESULTS SUMMARY

Performance Across Thresholds:
 threshold  total_detected  true_positives  false_positives  precision
      0.05              36               0               36        0.0
      0.10              29               0               29        0.0
      0.15              19               0               19        0.0
      0.20              16               0               16        0.0
      0.25              11               0               11        0.0
      0.30               8               0                8        0.0
      0.35               8               0                8        0.0
      0.40               7               0                7        0.0
      0.45               7               0                7        0.0
      0.50               7               0                7        0.0

RECOMMENDATIONS

Maximum Detection:
  Threshold: 0.05
  True positives: 0
  Precision: 0.000 (0.0%)

No threshold achieved precision >= 30%

No threshold achieved precis